In [5]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/real_estate_cleaned.csv', parse_dates=['date'])

q25, q75 = df['price'].quantile([0.25, 0.75])
df['price_segment'] = np.where(df['price'] <= q25, 'Low-cost',
                        np.where(df['price'] <= q75, 'Mid-market', 'Premium'))
print(df['price_segment'].value_counts(normalize=True))

num_cols = ['sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms', 'floors', 'view',
            'condition', 'waterfront', 'sqft_above', 'sqft_basement',
            'has_basement']
cat_cols = ['zipcode']         

X = df[num_cols + cat_cols]
y = df['price_segment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

price_segment
Mid-market    0.499670
Low-cost      0.250385
Premium       0.249945
Name: proportion, dtype: float64


In [6]:

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10), cat_cols),
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

fitted = {}
for name, m in models.items():
    fitted[name] = Pipeline([('pre', pre), ('model', m)]).fit(X_train, y_train)



In [7]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

labels = ['Low-cost', 'Mid-market', 'Premium']
for name, p in fitted.items():
    pred = p.predict(X_test)
    print(name, '| accuracy', round(accuracy_score(y_test, pred), 3),
          '| macro F1', round(f1_score(y_test, pred, average='macro'), 3))

best_pred = fitted['Logistic Regression'].predict(X_test)
print(confusion_matrix(y_test, best_pred, labels=labels))
print(classification_report(y_test, best_pred))

Logistic Regression | accuracy 0.826 | macro F1 0.825
Random Forest | accuracy 0.789 | macro F1 0.786
[[187  41   0]
 [ 40 383  31]
 [  0  46 181]]
              precision    recall  f1-score   support

    Low-cost       0.82      0.82      0.82       228
  Mid-market       0.81      0.84      0.83       454
     Premium       0.85      0.80      0.82       227

    accuracy                           0.83       909
   macro avg       0.83      0.82      0.83       909
weighted avg       0.83      0.83      0.83       909

